# 5.3 · multi-hierarchy kv-transfer — is the sub-**sub**-goal carried the same way?

Paper weak point: §5 (NRGS) shifts the hierarchy level, while the rest of the paper fixes it at the sub-goal level; Figure 1 claims *different levels of the hierarchy are maintained the same way* without a direct test.

Here we repeat the **kv-transfer (task-transfer)** setup one level down. With 2 sub-goals:

```
p_src: a'b -> b        p_dst: a'c -> c        kv-transfer target: b
```
Now let the frontier sub-goals be **conjunctions** of sub-sub-goals that share a prefilled first conjunct `x`:
```
p_src: a'(x'y) -> y    p_dst: a'(x'z) -> z    kv-transfer target: y
```
e.g. `words: list[str]` —
`b` = words that start with "b" **and are palindromes**, `c` = words that start with "b" **and have more than 3 characters**;
both prefill `b = [w for w in words if w.startswith("b") and` → the pivot is the **`and`** token.

The intervention is otherwise identical to kv-transfer (`src.diff.functional.run_kv_transfer_intervention`, the paper's multi-predicate config: goal-250 patch/readout heads, α=2, `pad` alignment, no ablation): cache SRC's task-description key+value on the patch heads and transplant it onto DST's. Success = patched DST writes `y` (SRC's sub-sub-goal) instead of `z`. GoalScope read-outs are taken at the `and` pivot (plus, for contrast, at the sub-goal-level `=` of the clean runs). py→py only. Qualitative — no oracle by default.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import json, re
from dataclasses import dataclass, field
import torch, numpy as np
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
import src.diff.data as D                   # dataset generators

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape

with open(os.path.join(os.path.dirname(env_utils.DEFAULT_RESULTS_DIR),
                       "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json")) as f:
    GOAL_HEADS_250 = [(l, h) for l, h in json.load(f)]

## Knobs

`SPAN` — which DESCRIPTION span is transplanted (k+v, SRC → DST):
- `"list"` — the whole rendered task list `(1) a … (2) b …` (= the multi-predicate kv-transfer convention, `D.materialize`);
- `"frontier"` — only the frontier sub-goal's description (`the list of words … that start with "b" and are palindromes`);
- `"clause"` — only the distinguishing sub-sub-goal phrase (`are palindromes` → `have more than 3 characters`).

In [ ]:
#------------------------------------------------------------------------------------------------------
ALPHA = 2.0                 # OV-write boost of PATCH_HEADS at the pivot on the PATCHED gen/readout (1.0 = off)
ALIGN_KNOB = "pad"          # "pad" = left-pad the shorter span so the dest span is fully overwritten | "right"
SPAN = "list"               # "list" | "frontier" | "clause"   (see above)
PATCH_HEADS = GOAL_HEADS_250
READOUT_HEADS = GOAL_HEADS_250
MAX_NEW = 48
#------------------------------------------------------------------------------------------------------

## Instances — a `MultiPredicateInstance` whose frontier prefill runs past `b =` up to the shared conjunct's `and`

`EXAMPLES` holds the hand-written conjunction tasks: a fully-prefilled sub-goal `a`, a shared first conjunct `x` (prefilled), and the two alternative second conjuncts `y` (SRC) / `z` (DST). `kw` are regexes used only for a quick qualitative verdict on the first line the model writes.

In [ ]:
@dataclass(frozen=True)
class HierInstance(D.MultiPredicateInstance):
    sub_prefill: str = ""                          # the shared conjunct x, prefilled after `b =` up to `and`
    clauses: dict = field(default_factory=dict)    # frontier id -> its distinguishing sub-sub-goal phrase
    span: str = "list"                             # transplanted span: "list" | "frontier" | "clause"

    def _parts(self, lang, frontier):
        q, pf = super()._parts(lang, frontier)     # letters + var_eq stub -> prefill ends at "    b ="
        return q, pf + " " + self.sub_prefill

    def full_task_text(self, frontier):            # D.materialize reads the k/v span from here
        if self.span == "frontier":
            return frontier.description
        if self.span == "clause":
            return self.clauses[frontier.id]
        return super().full_task_text(frontier)


EXAMPLES = [
    dict(name="words_b", dt="list[str]", var="words",
         a=("the number of words", "{res} = len({arg})"),
         template='the list of words in `words` that {x} and {c}', x='start with "b"',
         prefill='[w for w in words if w.startswith("b") and',
         y=dict(id="palindrome", nl="are palindromes", kw=r"\[::-1\]"),
         z=dict(id="len_gt3", nl="have more than 3 characters", kw=r"len\(w\)\s*>")),
    dict(name="words_long", dt="list[str]", var="words",
         a=("the number of distinct words", "{res} = len(set({arg}))"),
         template='the list of words in `words` that {x} and {c}', x="have more than 3 characters",
         prefill="[w for w in words if len(w) > 3 and",
         y=dict(id="upper", nl="are all uppercase", kw=r"isupper\(\)"),
         z=dict(id="has_e", nl='contain the letter "e"', kw=r"['\"]e['\"]\s+in")),
    dict(name="nums_even", dt="list[int]", var="nums",
         a=("the sum of the numbers", "{res} = sum({arg})"),
         template="the list of numbers in `nums` that {x} and {c}", x="are even",
         prefill="[x for x in nums if x % 2 == 0 and",
         y=dict(id="gt10", nl="are greater than 10", kw=r">\s*10"),
         z=dict(id="neg", nl="are negative", kw=r"<\s*0")),
    dict(name="chars_alpha", dt="str", var="s",
         a=("the length of the string", "{res} = len({arg})"),
         template="the list of characters in `s` that {x} and {c}", x="are letters",
         prefill="[ch for ch in s if ch.isalpha() and",
         y=dict(id="upper", nl="are uppercase", kw=r"isupper\(\)"),
         z=dict(id="vowel", nl="are vowels", kw=r"aeiou")),
    dict(name="int_even", dt="int", var="n",
         a=("the absolute value of the number", "{res} = abs({arg})"),
         template="whether the number is {x} and {c}", x="even",
         prefill="n % 2 == 0 and",
         y=dict(id="pos", nl="positive", kw=r"n\s*>\s*0"),
         z=dict(id="div3", nl="divisible by 3", kw=r"%\s*3")),
]


def make_instance(ex, flip=False, span=None):
    # SRC pursues `y`, DST pursues `z` (swapped with flip=True)
    y, z = (ex["z"], ex["y"]) if flip else (ex["y"], ex["z"])
    front = lambda c: D.Predicate(id=c["id"], description=ex["template"].format(x=ex["x"], c=c["nl"]),
                                  result_name=c["id"], impl={"python": "", "cpp": ""})
    src_f, dst_f = front(y), front(z)
    a = D.Predicate(id="a", description=ex["a"][0], result_name="a", impl={"python": ex["a"][1], "cpp": ""})
    return HierInstance(
        datatype=ex["dt"], shared=(a,), src_frontier=src_f, dst_frontier=dst_f, var_name=ex["var"],
        presentation="listed", src_lang="python", dst_lang="python", naming="letters", letter_stub="var_eq",
        meta=dict(datatype=ex["dt"], example=ex["name"], flip=flip),
        sub_prefill=ex["prefill"], clauses={src_f.id: y["nl"], dst_f.id: z["nl"]}, span=span or SPAN,
    ), y, z

## Render one instance — SRC/DST prompts, the transplanted span (orange) and the `and` pivot

In [ ]:
EX_IDX, FLIP = 0, False

inst, Y, Z = make_instance(EXAMPLES[EX_IDX], flip=FLIP)
m = D.materialize(inst)
S0, T0 = token_span(m["src_prompt"], m["src_task_text"]), token_span(m["dst_prompt"], m["dst_task_text"])
src_tok, dst_tok, SRC_POS, DST_POS, SRC_SPAN, DST_SPAN = align_kv_spans(
    mt, m["src_prompt"], m["dst_prompt"], S0, T0, mode=ALIGN_KNOB,
    src_text=m["src_task_text"], dst_text=m["dst_task_text"])
for name, tok, span, pos in [("SRC", src_tok, SRC_SPAN, SRC_POS), ("DST", dst_tok, DST_SPAN, DST_POS)]:
    last = mt.tokenizer.decode(tok["input_ids"][0, -1])
    show_spans(tok, f"{name} · span={SPAN} {tuple(span)} · pivot token {last!r}",
               task_span=span, kv_span=(pos[0], pos[-1] + 1), piv=-1)
    assert last.strip() == "and", f"{name} pivot is {last!r}, expected 'and'"
print(f"SRC sub-sub-goal y = {Y['nl']!r}   |   DST sub-sub-goal z = {Z['nl']!r}")

## Intervention — transplant SRC's description k+v onto DST, read out at `and`

`verdict` is a regex check on the first line the patched model writes (`y` = SRC's sub-sub-goal = success, `z` = DST's native = no-op). The extra GoalScope read-outs at the clean runs' `=` token show what the goal heads carry one level up (the whole sub-goal `b`), for contrast with the `and` read-out (the next sub-sub-goal).

In [ ]:
from src.diff.functional import run_kv_transfer_intervention


def eq_pos(prompt):
    # index of the sub-goal-level `=` (the frontier's `b =`): the last '=' token before the pivot
    ids = prepare_input(prompts=prompt, tokenizer=mt)["input_ids"][0].tolist()
    return max(i for i, t in enumerate(ids) if mt.tokenizer.decode([t]).strip() == "=")


def verdict(gen, y, z):
    first = gen.strip().split("\n")[0]
    hy, hz = bool(re.search(y["kw"], first)), bool(re.search(z["kw"], first))
    return "y (SRC)" if hy and not hz else "z (DST)" if hz and not hy else "both" if hy else "neither"


def run_one(ex, flip=False, span=None, verbose=True, eq_readout=True):
    inst, y, z = make_instance(ex, flip=flip, span=span)
    out = run_kv_transfer_intervention(mt=mt, inst=inst, patch_heads=PATCH_HEADS, readout_heads=READOUT_HEADS,
                                       max_new=MAX_NEW, alpha=ALPHA, align=ALIGN_KNOB)
    v = {k: verdict(out[f"gen_{k}"], y, z) for k in ("src", "dst", "patched")}
    if verbose:
        rich_print(f"[bold]{ex['name']}{' (flipped)' if flip else ''} · span={inst.span}[/bold]   "
                   f"y(SRC)={y['nl']!r}  z(DST)={z['nl']!r}")
        rich_print(f"  [red1]clean SRC[/red1]   [{v['src']}] : {escape(out['gen_src'].split(chr(10))[0])}")
        rich_print(f"  [sea_green3]clean DST[/sea_green3]   [{v['dst']}] : {escape(out['gen_dst'].split(chr(10))[0])}")
        rich_print(f"  [salmon1]patched DST[/salmon1] [{v['patched']}] : {escape(out['gen_patched'].split(chr(10))[0])}")
        rich_print(f"  goalscope @ `and` · SRC    : [italic]{escape(out['ro_src'])}[/italic]")
        rich_print(f"  goalscope @ `and` · DST    : [italic]{escape(out['ro_dst'])}[/italic]")
        rich_print(f"  goalscope @ `and` · patched: [italic]{escape(out['ro_patched'])}[/italic]")
        if eq_readout:
            for side, pr in (("SRC", out["src_prompt"]), ("DST", out["dst_prompt"])):
                goalscope_readout(f"  goalscope @ `=`   · {side}   ", pr, eq_pos(pr), heads=READOUT_HEADS)
    return dict(example=ex["name"], flip=flip, span=inst.span, **{f"verdict_{k}": x for k, x in v.items()}, **out)


res = run_one(EXAMPLES[EX_IDX], flip=FLIP)

## All examples × both directions (compact)

In [ ]:
rows = []
for ex in EXAMPLES:
    for flip in (False, True):
        rows.append(run_one(ex, flip=flip, eq_readout=False))

from rich.table import Table
t = Table(title=f"multi-hierarchy kv-transfer · span={SPAN} · α={ALPHA} · {len(PATCH_HEADS)} heads")
for c in ("example", "SRC→y", "clean SRC", "clean DST", "patched DST", "goalscope@and · patched"):
    t.add_column(c)
for r in rows:
    ex = next(e for e in EXAMPLES if e["name"] == r["example"])
    y = (ex["z"] if r["flip"] else ex["y"])["nl"]
    t.add_row(r["example"], y, r["verdict_src"], r["verdict_dst"], r["verdict_patched"], escape(r["ro_patched"][:80]))
rich_print(t)
print(f"patched -> y (SRC): {sum(r['verdict_patched'] == 'y (SRC)' for r in rows)}/{len(rows)}")

## Span ablation — transplant the whole list vs only the frontier description vs only the sub-sub-goal clause

In [ ]:
for span in ("list", "frontier", "clause"):
    run_one(EXAMPLES[EX_IDX], flip=FLIP, span=span, eq_readout=False)